In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: A Qiskit Function by Qedma

*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Fungsi Qiskit adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API Platform IBM Quantum). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.

## Gambaran keseluruhan
Walaupun unit pemprosesan kuantum telah bertambah baik dengan pesat dalam beberapa tahun kebelakangan ini, ralat akibat hingar dan ketidaksempurnaan perkakasan sedia ada tetap menjadi cabaran utama bagi pembangun algoritma kuantum. Apabila bidang ini mendekati pengiraan kuantum skala utiliti yang tidak boleh disahkan secara klasik, penyelesaian untuk membatalkan hingar dengan ketepatan yang terjamin semakin penting. Untuk mengatasi cabaran ini, Qedma telah membangunkan Quantum Error Mitigation (QESEM), yang disepadukan dengan lancar pada Platform IBM Quantum sebagai [Fungsi Qiskit](/guides/functions).

Dengan QESEM, pengguna boleh menjalankan litar kuantum mereka pada QPU berbising untuk mendapatkan keputusan tepat bebas ralat dengan overhed masa QPU yang sangat cekap, hampir kepada had asas. Untuk mencapai ini, QESEM memanfaatkan suite kaedah proprietari yang dibangunkan oleh Qedma untuk pencirian dan pengurangan ralat. Teknik pengurangan ralat merangkumi pengoptimuman gate, pentranspilan peka-hingar, penindasan ralat (ES), dan mitigasi ralat tidak berat sebelah (EM). Dengan gabungan kaedah berasaskan pencirian ini, pengguna boleh mencapai keputusan yang boleh dipercayai dan bebas ralat untuk litar kuantum bervolum besar yang generik, membuka kunci aplikasi yang tidak dapat dicapai sebaliknya.

Untuk penerangan penuh komponen asas, serta demonstrasi skala utiliti, rujuk kertas [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Penerangan
Anda boleh menggunakan fungsi QESEM oleh Qedma untuk menganggarkan dan melaksanakan litar anda dengan mudah menggunakan penindasan dan mitigasi ralat, mencapai volum litar yang lebih besar dan ketepatan yang lebih tinggi. Untuk menggunakan QESEM, anda menyediakan litar kuantum, satu set boleh cerap untuk diukur, ketepatan statistik sasaran untuk setiap boleh cerap, dan QPU yang dipilih. Sebelum menjalankan litar ke ketepatan sasaran, anda boleh menganggarkan masa QPU yang diperlukan berdasarkan pengiraan analitik yang tidak memerlukan pelaksanaan litar. Setelah anda berpuas hati dengan anggaran masa QPU, anda boleh melaksanakan litar dengan QESEM.

Apabila anda melaksanakan litar, QESEM menjalankan protokol pencirian peranti yang disesuaikan dengan litar anda, menghasilkan model hingar yang boleh dipercayai untuk ralat yang berlaku dalam litar. Berdasarkan pencirian, QESEM pertama melaksanakan pentranspilan peka-hingar untuk memetakan litar input ke set qubit dan gate fizikal, yang meminimumkan hingar yang mempengaruhi boleh cerap sasaran. Ini termasuk gate yang tersedia secara asli (CX/CZ pada peranti IBM&reg;), serta gate tambahan yang dioptimumkan oleh QESEM, membentuk set gate lanjutan QESEM. QESEM kemudian menjalankan set litar ES dan EM berasaskan pencirian pada QPU dan mengumpul hasil pengukuran mereka. Ini kemudian diproses secara klasik untuk memberikan nilai jangkaan tidak berat sebelah dan bar ralat untuk setiap boleh cerap, sepadan dengan ketepatan yang diminta.

![Gambaran keseluruhan Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM telah terbukti memberikan keputusan berketetapan tinggi untuk pelbagai aplikasi kuantum dan pada volum litar terbesar yang boleh dicapai hari ini. QESEM menawarkan ciri-ciri berikut untuk pengguna, yang ditunjukkan dalam bahagian penanda aras di bawah:
-	**Ketepatan terjamin:** QESEM mengeluarkan anggaran tidak berat sebelah untuk nilai jangkaan boleh cerap. Kaedah EM-nya dilengkapi dengan jaminan teori, yang - bersama dengan pencirian terkini Qedma - memastikan mitigasi menumpu ke output litar tanpa hingar sehingga ketepatan yang ditentukan pengguna. Berbanding dengan banyak kaedah EM heuristik yang terdedah kepada ralat sistematik atau berat sebelah, ketepatan terjamin QESEM penting untuk memastikan keputusan yang boleh dipercayai dalam litar dan boleh cerap kuantum yang generik.
-	**Penskalaan ke QPU besar:** Masa QPU QESEM bergantung pada volum litar, tetapi bebas daripada bilangan qubit. Qedma telah menunjukkan QESEM pada peranti kuantum terbesar yang tersedia hari ini, termasuk peranti IBM Quantum Eagle 127-qubit dan Heron 133-qubit.
-	**Bebas aplikasi:** QESEM telah ditunjukkan pada pelbagai aplikasi, termasuk simulasi Hamiltonian, VQE, QAOA, dan anggaran amplitud. Pengguna boleh memasukkan sebarang litar kuantum dan boleh cerap untuk diukur, dan mendapatkan keputusan tepat bebas ralat. Satu-satunya batasan didiktekan oleh spesifikasi perkakasan dan masa QPU yang diperuntukkan, yang menentukan volum litar yang boleh diakses dan ketepatan output. Berbanding, banyak penyelesaian pengurangan ralat adalah khusus untuk aplikasi atau melibatkan heuristik yang tidak terkawal, menjadikannya tidak terpakai untuk litar dan aplikasi kuantum generik.
-  **Set gate lanjutan:** QESEM menyokong gate dengan sudut pecahan, dan menyediakan gate $Rzz(\theta)$ sudut pecahan yang dioptimumkan Qedma pada peranti IBM Quantum Heron dan Eagle. Set gate lanjutan ini membolehkan kompilasi yang lebih cekap dan membuka kunci volum litar yang lebih besar sebanyak faktor 2 berbanding kompilasi CX/CZ lalai.
-	**Boleh cerap multiasas:** QESEM menyokong boleh cerap input yang terdiri daripada banyak rentetan Pauli yang tidak bertukar ganti, seperti Hamiltonian generik. Pilihan asas pengukuran dan pengoptimuman peruntukan sumber QPU (tembakan dan litar) kemudian dilakukan secara automatik oleh QESEM untuk meminimumkan masa QPU yang diperlukan untuk ketepatan yang diminta. Pengoptimuman ini, yang mengambil kira kesetiaan perkakasan dan kadar pelaksanaan, membolehkan anda menjalankan litar yang lebih dalam dan mendapatkan ketepatan yang lebih tinggi.
## Penanda aras
QESEM telah diuji pada pelbagai kes penggunaan dan aplikasi. Contoh-contoh berikut boleh membantu anda menilai jenis beban kerja yang boleh anda jalankan dengan QESEM.

Angka merit utama untuk mengukur kesukaran mitigasi ralat dan simulasi klasik untuk litar dan boleh cerap tertentu ialah **volum aktif**: bilangan gate CNOT yang mempengaruhi boleh cerap dalam litar. Volum aktif bergantung pada kedalaman dan lebar litar, pada wajaran boleh cerap, dan pada struktur litar, yang menentukan kon cahaya boleh cerap. Untuk maklumat lanjut, lihat ucapan dari [Sidang Kemuncak IBM Quantum 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM memberikan nilai yang sangat besar dalam rejim volum tinggi, memberikan keputusan yang boleh dipercayai untuk litar dan boleh cerap generik.

![Volum aktif](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikasi    | Bilangan qubit | Peranti | Penerangan litar | Ketepatan | Jumlah masa | Penggunaan runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Litar VQE  | 8              | Eagle (r3) | 21 lapisan jumlah, 9 asas pengukuran, rantai 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 lapisan unik x 3 langkah, topologi hex-berat 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 lapisan unik x 8 langkah, topologi hex-berat 2D                     | 97%      | 116 min      | 23 min          |
| Simulasi Hamiltonian Trotterized   | 40  | Eagle (r3)            | 2 lapisan unik x 10 langkah Trotter, rantai 1D                    | 97%      | 3 jam     | 25 min         |
| Simulasi Hamiltonian Trotterized   | 119  | Eagle (r3)           | 3 lapisan unik x 9 langkah Trotter, topologi hex-berat 2D                    | 95%      | 6.5 jam     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 lapisan unik x 15 langkah, topologi hex-berat 2D                    | 99%      | 52 min             | 9 min           |

Ketepatan diukur di sini relatif terhadap nilai ideal boleh cerap: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, di mana '$\epsilon$' ialah ketepatan mutlak mitigasi (ditetapkan oleh input pengguna), dan $\langle O \rangle_{ideal}$ ialah boleh cerap pada litar tanpa hingar.
'Penggunaan runtime' mengukur penggunaan penanda aras dalam mod kelompok (jumlah penggunaan kerja individu), manakala 'jumlah masa' mengukur penggunaan dalam mod sesi (masa dinding eksperimen), yang merangkumi masa klasik dan komunikasi tambahan. QESEM tersedia untuk pelaksanaan dalam kedua-dua mod, supaya pengguna boleh memanfaatkan sebaik-baiknya sumber yang ada.

Litar Kicked Ising 28-qubit mensimulasikan Quasicrystal Masa Diskret yang dikaji oleh Shinjo et al. (lihat [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) dan [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pada tiga gelung bersambung ibm_kawasaki. Parameter litar yang diambil di sini ialah $(\theta_x, \theta_z) = (0.9 \pi, 0)$, dengan keadaan awal feromagnet $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Boleh cerap yang diukur ialah nilai mutlak kemagnetan $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Eksperimen Kicked Ising skala utiliti dijalankan pada 136 qubit terbaik ibm_fez; penanda aras khusus ini dijalankan pada sudut Clifford $(\theta_x, \theta_z) = (\pi, 0)$, di mana volum aktif berkembang perlahan dengan kedalaman litar, yang - bersama dengan kesetiaan peranti yang tinggi - membolehkan ketepatan tinggi pada masa runtime yang singkat.

Litar simulasi Hamiltonian Trotterized adalah untuk model Ising Medan Transversal pada sudut pecahan: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ dan $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ secara berturutan (lihat [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Litar skala utiliti dijalankan pada 119 qubit terbaik ibm_brisbane, manakala eksperimen 40-qubit dijalankan pada rantai terbaik yang tersedia. Ketepatan dilaporkan untuk kemagnetan; keputusan berketetapan tinggi diperoleh untuk boleh cerap berwajaran lebih tinggi juga.

Litar VQE dibangunkan bersama penyelidik dari Pusat Teknologi dan Aplikasi Kuantum di Deutsches Elektronen-Synchrotron (DESY). Boleh cerap sasaran di sini ialah Hamiltonian yang terdiri daripada sejumlah besar rentetan Pauli yang tidak bertukar ganti, menekankan prestasi QESEM yang dioptimumkan untuk boleh cerap pelbagai asas. Mitigasi diterapkan pada ansatz yang dioptimumkan secara klasik; walaupun keputusan ini masih belum diterbitkan, keputusan dengan kualiti yang sama akan diperoleh untuk litar berbeza dengan sifat struktur yang serupa.
## Mulakan
Sahkan menggunakan [kunci API Platform IBM Quantum](http://quantum.cloud.ibm.com/) anda, dan pilih Fungsi Qiskit QESEM seperti berikut. (Petikan kod ini menganggap anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan.)

In [ ]:
import qiskit
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

### Time estimation job example

The time estimation job is useful for estimating the QPU time required for a given `pub` and `backend_name`. `backend_name` can also be set to any simulator backend; for example, `fake_fez`.

QESEM uses a quasi-probabilistic, characterization-based EM method. This method has a QPU time overhead that roughly scales as:

$T_{QPU} = a \frac{e^{\alpha IF\cdot V_a}}{\epsilon^2} + b$

Where $V_a$ is the active volume of the circuit, $\epsilon$ is the target precision, and $IF$ is the infidelity of the native gates.

Note that `"estimate_time_only": "empirical"` uses a few minutes of QPU time to estimate the time required for the job (if the backend is a real device; if it's a simulator, then no QPU time is used). This will usually take around 5 minutes, but no more than 10 minutes. If the infidelity changes drastically between the empirical time estimation job and the mitigation job, the QPU time will also change drastically.

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
backend_name = "fake_fez"

circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "empirical",
    },
    backend_name=backend_name,  # example: "fake_fez", "ibm_fez"
)

In [4]:
time_estimate_result = (
    time_estimation_job.result()
)  # a list of results per pub (circuit)

Petikan kod berikut menunjukkan cara mendapatkan semula anggaran masa QPU (`estimate_time_only` ditetapkan):

In [106]:
pub_result = time_estimate_result[0]

print(
    f"The estimated QPU time for mitigation for this PUB is: {pub_result.metadata['time_estimation_sec']}"
)
print(
    f"The QPU time that this time estimation job took is (here it is 0 because we used fake_fez): {pub_result.metadata['total_qpu_time']}"
)
print(
    f"Gates fidelity measured during the experiment: {pub_result.metadata['gate_fidelities']}"
)
print(f"Total shots: {pub_result.metadata['total_shots']}")
print(f"Resource usage breakdown: {pub_result.metadata['resource_usage']}")

The estimated QPU time for mitigation for this PUB is: 300
The QPU time that this time estimation job took is (here it is 0 because we used fake_fez): 0
Gates fidelity measured during the experiment: {'CZ': 0.9951354916722668, 'ID1Q': 0.9991246627329172}
Total shots: 220000
Resource usage breakdown: {'RUNNING: MAPPING': {'CPU_TIME': 33.6066133165732, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 184.53575124032795, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}


Petikan kod berikut menunjukkan cara mendapatkan semula keputusan mitigasi (apabila `estimate_time_only` tidak ditetapkan) dan metrik pelaksanaan. Ini mengandungi data penting yang membolehkan pemahaman lebih mendalam tentang bagaimana parameter berbeza mempengaruhi pelaksanaan QESEM. Ia juga mungkin relevan apabila menulis kertas berdasarkan penyelidikan anda.

In [6]:
empirical_estimation_mitigation_results = time_estimate_result[0].metadata[
    "empirical_estimation_mitigation_results"
][0]  # a list per parameter

In [12]:
print("Partial results for the observables:")

print(
    f"    Mitigated expectation values: {empirical_estimation_mitigation_results.data.evs}"
)
print(
    f"    Mitigated error bars: {empirical_estimation_mitigation_results.data.stds}"
)
print(
    f"    Number of shots used for mitigation: {empirical_estimation_mitigation_results.metadata['mitigation_shots']}"
)
transpiled_circ = empirical_estimation_mitigation_results.metadata[
    "transpiled_circ"
]
print(f"    Qubit mapping: {transpiled_circ['qubit_maps']}")
print(
    f"    Number of measurement bases: {transpiled_circ['num_measurement_bases']}\n"
)

# results per obs
emp_obs_results = empirical_estimation_mitigation_results.metadata["results"][
    0
]
# print(f"Results for each observable: {results}")
print("Results for each observable:")

for i, (obs_array, result_dict) in enumerate(emp_obs_results):
    # obs_array, result_dict = results
    print(f"Observable {i+1}: {obs_array}")
    print(
        f"    QESEM mitigated value: {result_dict['qesem']['value']} \u00b1 {result_dict['qesem']['error_bar']}"
    )

Partial results for the observables:
    Mitigated expectation values: [1.00347302 1.00693905]
    Mitigated error bars: [0.00304061 0.00714276]
    Number of shots used for mitigation: 180000
    Qubit mapping: [[[0, 136], [1, 143], [2, 142], [3, 141], [4, 140]]]
    Number of measurement bases: 2

Results for each observable:
Observable 1: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
    QESEM mitigated value: 1.003473015776871 ± 0.0030406128032204015
Observable 2: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
    QESEM mitigated value: 1.0069390542613554 ± 0.0071427606736885925


### QESEM mitigation job example

The following example executes a QESEM job:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # example: "ibm_fez"
    # options = {
    #     "estimate_time_only": "empirical",
    #     "default_precision": 0.2,  # Default precision is applied to all pubs that don't have a precision specified, see API reference for more details
    #     "max_execution_time": 3600,  # You can specify a maximum QPU time in seconds, see API reference for more details
    #     "transpilation_level": "standard",  # "minimal_with_layout_opt" for minimal transpilation, see API reference for more details
    #     "parallel_execution": True,  # True for parallel execution, see API reference for more details
    # },
)

## Dapatkan sokongan

Pasukan sokongan Qedma sedia membantu! Jika anda menghadapi sebarang isu atau mempunyai soalan tentang penggunaan Fungsi Qiskit QESEM, jangan teragak-agak untuk menghubungi kami. Kakitangan sokongan kami yang berpengetahuan dan mesra bersedia membantu anda dengan sebarang kebimbangan teknikal atau pertanyaan yang anda miliki.

Anda boleh menghantar e-mel kepada kami di support@qedma.com untuk mendapatkan bantuan. Sila sertakan seberapa banyak maklumat tentang isu yang anda alami untuk membantu kami memberikan respons yang cepat dan tepat. Anda juga boleh menghubungi wakil Qedma POC khusus anda melalui e-mel atau telefon.

Untuk membantu kami membantu anda dengan lebih cekap, sila berikan maklumat berikut apabila menghubungi kami:

- Penerangan terperinci tentang isu tersebut
- ID kerja
- Sebarang mesej atau kod ralat yang berkaitan

Kami komited untuk memberikan anda sokongan yang segera dan berkesan untuk memastikan anda mendapat pengalaman terbaik dengan Fungsi Qiskit kami.

Kami sentiasa berusaha untuk meningkatkan produk kami dan kami mengalu-alukan cadangan anda! Jika anda mempunyai idea tentang cara kami boleh meningkatkan perkhidmatan atau ciri yang anda ingin lihat, sila hantar pemikiran anda kepada support@qedma.com.

## Langkah seterusnya

> **Tip:** - [Minta akses ke Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Lawati [rujukan API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) untuk Fungsi Qiskit ini.
> - Semak [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Semak [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).

In [25]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)
print(sample_job.status())
sample_result = sample_job.result()

3ac6b2df-15b0-4dc0-8f48-cf14bd20a1c8
DONE


The following code snippet demonstrates how to retrieve the mitigation results and execution metrics. These contain essential data that enables a deeper understanding of how different parameters impact the QESEM execution. It may also be relevant when writing a paper based on your research.

In [50]:
for pub_idx, pub_result in enumerate(
    sample_result
):  # each element in the list is a result for a different pub, here we sent only one pub
    print(f"\nPUB {pub_idx}:")
    print(
        f"  The QPU time that this job took is (here it is 0 because we used fake_fez): {pub_result.metadata['total_qpu_time']}"
    )
    print(
        f"  Gates fidelity measured during the experiment: {pub_result.metadata['gate_fidelities']}"
    )
    print(f"  Total shots: {pub_result.metadata['total_shots']}")
    print(
        f"  Number of shots used for mitigation: {pub_result.metadata['mitigation_shots']}"
    )
    print(
        f"  Resource usage breakdown: {pub_result.metadata['resource_usage']}"
    )


PUB 0:
  The QPU time that this job took is (here it is 0 because we used fake_fez): 0.0
  Gates fidelity measured during the experiment: {'CZ': 0.9953704216147041, 'ID1Q': 0.9991834123567518}
  Total shots: 446000
  Number of shots used for mitigation: 194000
  Resource usage breakdown: {'RUNNING: MAPPING': {'CPU_TIME': 32.52745003718883, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 257.850521848537, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}


In `metadata["results"]`, results are grouped first by circuit instance and then by observable.

In [99]:
print("Full QESEM mitigation results:")

for pub_idx, pub_result in enumerate(sample_result):
    print(f"\nPUB {pub_idx}:")

    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata.get("noisy_results")
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")

    print("  Transpiled circuits:")
    for circ_idx, transpiled_circ in enumerate(
        pub_result.metadata["transpiled_circs"]
    ):
        print(f"    Circuit {circ_idx}:")
        # print(f"      Circuit: \n {transpiled_circ['circuit']}") # not printing it because it's long but you can see the transpiled circuit itself
        print(f"      Qubit mapping: {transpiled_circ['qubit_maps']}")
        print(
            f"      Measurement bases: {transpiled_circ['num_measurement_bases']}"
        )

    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Full QESEM mitigation results:

PUB 0:
  Mitigated expectation values: [1.00648343 1.00636289]
  Mitigated error bars: [0.00253812 0.00693586]
  Unmitigated expectation values: [0.98031429 0.96357143]
  Unmitigated error bars: [0.00124128 0.00578812]
  Transpiled circuits:
    Circuit 0:
      Qubit mapping: [[[0, 140], [1, 141], [2, 142], [3, 143], [4, 136]]]
      Measurement bases: 2
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 1.0064834305181962 ± 0.002538119914534849
        Unmitigated value: 0.9803142857142859 ± 0.0012412835813609938
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 1.0063628870614818 ± 0.006935859820870656
        Unmitigated value: 0.9635714285714285 ± 0.005788121870526659


**Main results breakdown:**
- `mitigated`: the fully mitigated QESEM expectation value.
- `unmitigated`: the raw physical-noise result without error mitigation.

#### QESEM heuristic extrapolation results

In a standard QESEM run with a single `precision` float, the results also include automatically available noise-scaling points that are used for the QESEM heuristic. These points are computed without additional QPU resources.

Scale `1.0` represents the physical device noise level with readout mitigation (REM), while scale `2.0` is the complementary noise-amplified point, also with REM. These points are used to produce the `qesem_heuristic` result.

- `qesem_heuristic`: a ZNE-style estimate computed from the available noise-scaled data. Currently, this uses exponential extrapolation.
- `noise_scaling.results_with_REM`: expectation values at different noise scales, all with readout mitigation (REM).

A subtle but important detail is that the scale `1.0` result is not the same as the `unmitigated` result. Both correspond to the physical device noise level, but the scale `1.0` point includes readout mitigation, while `unmitigated` does not.

In [101]:
print("QESEM heuristic results:")

for pub_idx, pub_result in enumerate(sample_result):
    print(f"\nPUB {pub_idx}:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"  Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"    Observable {obs_idx}: {obs_array}")
            qesem_heuristic = result_dict["qesem_heuristic"][0]
            print("      QESEM heuristic:")
            print(
                f"        Value: {qesem_heuristic['value']} ± {qesem_heuristic['error_bar']}"
            )
            print(
                f"        Extrapolation: {qesem_heuristic['extrapolation']}"
            )
            print(
                f"        Scale factors: {qesem_heuristic['scale_factors']}"
            )
            noise_scaling = result_dict["noise_scaling"]
            print("      Noise scaling results:")
            print(
                f"        Scaling method: {noise_scaling['scaling_method']}"
            )
            print("        Results with Readout mitigation (REM):")
            for rem_result in sorted(
                (
                    item
                    for item in noise_scaling["results_with_REM"]
                    if item["scale"] != 0.0
                ),
                key=lambda item: item["scale"],
            ):
                print(
                    f"          Scale factor {rem_result['scale']}: {rem_result['value']} ± {rem_result['error_bar']}"
                )

QESEM heuristic results:

PUB 0:
  Circuit 0:
    Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
      QESEM heuristic:
        Value: 1.0008161638888535 ± 0.0038859458884403964
        Extrapolation: exponential
        Scale factors: [1.0, 2.0]
      Noise scaling results:
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 1.0: 0.9918395459270772 ± 0.0012565417579355634
          Scale factor 2.0: 0.982943441922748 ± 0.0028919278067695018
    Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
      QESEM heuristic:
        Value: 0.9960853148925298 ± 0.013811635038961175
        Extrapolation: exponential
        Scale factors: [1.0, 2.0]
      Noise scaling results:
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 1.0: 0.9902860583785115 ± 0.005921236914723409
          Scale factor 2.0: 0.9845205654

<Admonition type="note">
The following examples focus on feature-specific inputs and results, so they do not print the full execution metrics each time. The top-level metadata shown earlier, such as <code>total_qpu_time</code>, <code>gate_fidelities</code>, <code>total_shots</code>, <code>mitigation_shots</code>, and <code>resource_usage</code>, is available for these jobs as well.

Some variables from the previous examples, including the backend, observables, and base circuit parameters, are reused below for conciseness.
</Admonition>

<Admonition type="note">
All of the following examples can also be run with empirical time estimation. To enable it, pass <code>"estimate_time_only": "empirical"</code> in the function options.
</Admonition>

### Parameterized circuit example

Many algorithms evaluate the same circuit at several parameter values. Sending the parameterized circuit as one QESEM job lets QESEM share characterization and calibration across the circuit instances, which can reduce QPU-time overhead compared with running separate jobs.

Submitting a parameterized circuit requires using the `"minimal_with_layout_opt"` transpilation level.
Circuits submitted at this level should already be expressed using the backend's basis gates, depending on the backend. At this level, QESEM keeps the submitted structure as close as possible to the input circuit, respects barriers during layerification (grouping operations into layers of parallel two-qubit gates), and still handles hardware mapping to high-fidelity qubits and device connectivity automatically.

In practice, this means you should transpile circuits to the target backend’s basis gates before submission. A simple basis-gate transpilation example is shown below.

QESEM currently supports only one observable per parameter set. The two parameter rows below are zipped with the two observables: the first row is measured with `avg_magnetization`, and the second row is measured with `other_observable`.

In [ ]:
# Transpile to the backend basis gates only. With minimal_with_layout_opt, QESEM handles hardware mapping/connectivity and observable layout internally.
from qiskit_ibm_runtime.fake_provider import FakeFez

backend = FakeFez()
basis = backend.operation_names
print(basis)

param0 = qiskit.circuit.Parameter("param0")
param1 = qiskit.circuit.Parameter("param1")
parametrized_circ = qiskit.QuantumCircuit(5)
parametrized_circ.rx(param0, 0)
parametrized_circ.rx(param1, 1)
parametrized_circ.cx(0, 1)
parametrized_circ.cx(2, 3)
parametrized_circ.cx(1, 2)
parametrized_circ.cx(3, 4)

parametrized_circ = qiskit.transpile(
    parametrized_circ, basis_gates=basis, optimization_level=1
)
parametrized_parameter_values = [[0.5, 0.1], [0.0, 0.6]]
parametrized_observables = [avg_magnetization, other_observable]

parametrized_job = qesem_function.run(
    pubs=[
        (
            parametrized_circ,
            parametrized_observables,
            parametrized_parameter_values,
            0.1,
        )
    ],
    backend_name=backend_name,
    options={
        "max_execution_time": 300,
        "transpilation_level": "minimal_with_layout_opt",
    },
)

In [72]:
print(parametrized_job.job_id)
print(parametrized_job.status())

d1b0e29b-196c-4896-aec6-44a268ebd874
DONE


In [73]:
parametrized_result = parametrized_job.result()

In [77]:
print("Parameterized circuit QESEM results:")

for pub_idx, pub_result in enumerate(parametrized_result):
    print(f"\nPUB {pub_idx}:")
    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each parameter value:")
    for param_idx, param_results in enumerate(pub_result.metadata["results"]):
        print(
            f"    Parameter set {param_idx}: {parametrized_parameter_values[param_idx]}"
        )
        for obs_idx, (obs_array, result_dict) in enumerate(param_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Parameterized circuit QESEM results:

PUB 0:
  Mitigated expectation values: [0.92392021 0.82517653]
  Mitigated error bars: [0.00565281 0.00616016]
  Unmitigated expectation values: [0.9028     0.78771429]
  Unmitigated error bars: [0.00335142 0.00925413]
  Results for each parameter value:
    Parameter set 0: [0.5, 0.1]
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 0.923920210615709 ± 0.005652811570890183
        Unmitigated value: 0.9028 ± 0.0033514176105045447
    Parameter set 1: [0.0, 0.6]
      Observable 0: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 0.8251765289285893 ± 0.006160161743353999
        Unmitigated value: 0.7877142857142858 ± 0.009254130564027902


### Multi-pub example

Multi-pub execution is useful when you want to run several related circuits in one QESEM job. Like parameterized execution, this can reduce overhead because QESEM can share characterization and calibration across circuit instances instead of repeating it in separate jobs.

This is especially useful for circuits with a shared layer structure, such as **Trotter-type** workloads, where different circuits reuse the same unique layers. In that case, running them together can reduce characterization cost compared with independent QESEM jobs.

Multi-pub jobs require `"transpilation_level": "minimal_with_layout_opt"`. As in the parameterized example, the circuits should be transpiled to the target backend’s basis gates before submission. QESEM then handles device connectivity, layout, and mapping to high-fidelity qubits internally.

Each PUB below contains one circuit and the same two observables used earlier in the notebook, so the returned `PrimitiveResult` contains one `PubResult` per input circuit.

The example below uses two simple Trotter circuits with the same layer pattern: `circ_a` has one Trotter layer, and `circ_b` repeats the same layer pattern twice. This makes the shared structure explicit.

In [ ]:
def make_trotter_circuit(num_qubits, num_layers, zz_angle=0.2, x_angle=0.1):
    trotter_circ = qiskit.QuantumCircuit(num_qubits)
    for _ in range(num_layers):
        for q in range(num_qubits):
            trotter_circ.rx(x_angle, q)
        trotter_circ.barrier()
        for q in range(0, num_qubits - 1, 2):
            trotter_circ.rzz(zz_angle, q, q + 1)
        trotter_circ.barrier()
        for q in range(1, num_qubits - 1, 2):
            trotter_circ.rzz(zz_angle, q, q + 1)
        trotter_circ.barrier()
    return trotter_circ


circ_a = make_trotter_circuit(num_qubits=5, num_layers=1)
circ_b = make_trotter_circuit(num_qubits=5, num_layers=2)

In [ ]:
multi_pubs = [
    (
        qiskit.transpile(qci, basis_gates=basis, optimization_level=1),
        [avg_magnetization, other_observable],
    )
    for qci in [circ_a, circ_b]
]

multi_circ_job = qesem_function.run(
    pubs=multi_pubs,
    backend_name=backend_name,
    options={
        "max_execution_time": 300,
        "transpilation_level": "minimal_with_layout_opt",
        "default_precision": 0.1,
    },
)

In [91]:
print(multi_circ_job.job_id)
print(multi_circ_job.status())

e34565b8-7262-4133-a120-de42ce624a99
DONE


In [92]:
multi_circ_result = multi_circ_job.result()

In [105]:
print("Multi-pub QESEM results:")

for pub_idx, pub_result in enumerate(multi_circ_result):
    print(f"\nPUB {pub_idx}:")
    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Multi-pub QESEM results:

PUB 0:
  Mitigated expectation values: [0.99502406 1.02209332]
  Mitigated error bars: [0.00483819 0.00707488]
  Unmitigated expectation values: [0.96934286 0.97271429]
  Unmitigated error bars: [0.00124855 0.00617294]
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 0.9950240647925642 ± 0.004838188086259301
        Unmitigated value: 0.9693428571428573 ± 0.0012485470362492692
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 1.0220933230604674 ± 0.007074884384355636
        Unmitigated value: 0.9727142857142859 ± 0.006172939375512439

PUB 1:
  Mitigated expectation values: [0.98850017 1.02555188]
  Mitigated error bars: [0.0077912  0.01672652]
  Unmitigated expectation values: [0.93682857 0.95371429]
  Unmitigated error bars: [0.00156245 0.00665735]
  Result

### Quasi-probabilistic Error Tuning (QET) example

Quasi-probabilistic Error Tuning (QET) requests expectation values at selected noise scale factors. This is useful for custom noise-scaling studies and Zero-Noise Extrapolation workflows. Scale `1.0` is the physical noise level, values between `0.0` and `1.0` partially reduce noise, and values above `1.0` amplify noise.

To use QET with the Qiskit Function, pass a dictionary as the PUB precision. The dictionary maps each requested noise scale to its target precision. Returned scale-factor results are stored in `noise_scaling.results_with_REM` and include readout mitigation. The scale `1.0` point is therefore not identical to the unmitigated value, because `1.0` includes readout mitigation while `unmitigated` does not.

When a scale is requested, QESEM also returns the complementary scale around `1.0` without extra QPU usage. For example, requesting `0.5` can also return `1.5`, and requesting `1.3` can also return `0.7`. The precision on the complementary scale is not guaranteed.

In [88]:
noise_scale_precision = {0.5: 0.15, 1.3: 0.2}

qet_job = qesem_function.run(
    pubs=[
        (
            circ,
            [avg_magnetization, other_observable],
            None,
            noise_scale_precision,
        )
    ],
    backend_name=backend_name,
    options={"max_execution_time": 300},
)

In [95]:
print(qet_job.job_id)
print(qet_job.status())

8195fa58-f037-4651-8715-36ce1cdc5521
DONE


In [96]:
qet_result = qet_job.result()

In [104]:
print("QET noise-scaling results:")

for pub_idx, pub_result in enumerate(qet_result):
    print(f"\nPUB {pub_idx}:")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            noise_scaling = result_dict["noise_scaling"]
            print(
                f"        Scaling method: {noise_scaling['scaling_method']}"
            )
            print("        Results with Readout mitigation (REM):")
            for rem_result in sorted(
                (
                    item
                    for item in noise_scaling["results_with_REM"]
                    if item["scale"] != 0.0
                ),
                key=lambda item: item["scale"],
            ):
                print(
                    f"          Scale factor {rem_result['scale']}: {rem_result['value']} ± {rem_result['error_bar']}"
                )

QET noise-scaling results:

PUB 0:
  Unmitigated expectation values: [0.97822857 0.96171429]
  Unmitigated error bars: [0.00123812 0.00672958]
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 0.5: 0.9938730340199383 ± 0.0032116907357568275
          Scale factor 0.7: 0.9963976191853445 ± 0.00036300258869586616
          Scale factor 1.0: 0.9898115079506586 ± 0.0012525947426560995
          Scale factor 1.3: 0.9864667065580341 ± 0.002633221613518526
          Scale factor 1.5: 0.9838755527197551 ± 0.002948417797996015
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 0.5: 1.0006538544450332 ± 0.002121742014777343
          Scale factor 0.7: 1.0004159

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Try the [Simulate 2D tilted-field Ising with the QESEM function](/docs/tutorials/qedma-2d-ising-with-qesem) tutorial.
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).
- Review [Aharonov, D., et al. (2025). Syndrome aware mitigation of logical errors. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).
- Review [Aharonov, D., et al. (2025). On the Importance of Error Mitigation for Quantum Computation. arXiv preprint arXiv:2512.23810](https://arxiv.org/abs/2512.23810).
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Goldack, M., et al. (2026). Computing Statistical Properties of Velocity Fields on Current Quantum Hardware. arXiv preprint arXiv:2601.10166](https://arxiv.org/pdf/2601.10166).
- Review [Sakuma, R., et al. (2026). Point-group symmetry analysis of many-electron wavefunctions on a quantum computer arXiv preprint arXiv:2605.24824](https://arxiv.org/abs/2605.24824).

</Admonition>